import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import os

df_ideo = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_mmxeu.csv')
del df_ideo['lrscale'], df_ideo['lrscale_mmx']
def best_classifier_iterative_corrected(df, target_column, feature_range):
    """
    Evalúa modelos de clasificación (RF, XGBoost, LightGBM, CatBoost, SVM) 
    con diferentes números de características y selección de características por modelo.

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        target_column (str): Nombre de la columna objetivo.
        feature_range (tuple): Rango de números de características a iterar (inicio, fin).
    """

    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)  # No solo la columna 1

            # Calcular AUC para multiclase usando 'ovr'
            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier_iterative_corrected.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_classifier_name

# Ejemplo de uso (asumiendo que df_ideo está definido):
best_model, best_k, best_features, best_classifier_name = best_classifier_iterative_corrected(df_ideo, 'ideologia', (10, 18))

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import os

df_ideo = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_mmxeu.csv')
del df_ideo['lrscale'], df_ideo['lrscale_mmx']

def bestclassf1(df, target_column, feature_range):
    """
    Evalúa modelos de clasificación (RF, XGBoost, LightGBM, CatBoost, SVM) 
    con diferentes números de características y selección de características por modelo.
    """

    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""
    best_metrics = {}

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)
            y_pred = classifier.predict(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted')
            recall = recall_score(y_test, y_pred, average='weighted')
            f1 = f1_score(y_test, y_pred, average='weighted')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name
                best_metrics = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1
                }

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestClass.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")
        f.write(f"Accuracy: {best_metrics['accuracy']}\n")
        f.write(f"Precision: {best_metrics['precision']}\n")
        f.write(f"Recall: {best_metrics['recall']}\n")
        f.write(f"F1-score: {best_metrics['f1']}\n")

    return best_model, best_k, best_features, best_classifier_name, best_metrics

# Ejemplo de uso (asumiendo que df_ideo está definido):
best_model, best_k, best_features, best_classifier_name, best_metrics = bestclassf1(df_ideo, 'ideologia', (8, 12))

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_metrics['accuracy']}")
print(f"Precision: {best_metrics['precision']}")
print(f"Recall: {best_metrics['recall']}")
print(f"F1-score: {best_metrics['f1']}")

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import os

df_ideo = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_mmxeu.csv')
del df_ideo['lrscale'], df_ideo['lrscale_mmx']

def bestclassf1(df, target_column, feature_range):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""
    best_metrics = {}

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)
            y_pred = classifier.predict(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted')
            recall = recall_score(y_test, y_pred, average='weighted')
            f1 = f1_score(y_test, y_pred, average='weighted')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name
                best_metrics = {
                    'auc': auc,
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1
                }

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestClassf1.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_metrics['auc']}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")
        f.write(f"Accuracy: {best_metrics['accuracy']}\n")
        f.write(f"Precision: {best_metrics['precision']}\n")
        f.write(f"Recall: {best_metrics['recall']}\n")
        f.write(f"F1-score: {best_metrics['f1']}\n")

    return best_model, best_k, best_features, best_classifier_name, best_metrics


best_model, best_k, best_features, best_classifier_name, best_metrics = bestclassf1(df_ideo, 'ideologia', (8, 12))

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"AUC: {best_metrics['auc']}")
print(f"Accuracy: {best_metrics['accuracy']}")
print(f"Precision: {best_metrics['precision']}")
print(f"Recall: {best_metrics['recall']}")
print(f"F1-score: {best_metrics['f1']}")

c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 68
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 8
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteratio

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 79
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 9
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

...

c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 101
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 11
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000025 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 112
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 12
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491
Mejor modelo: SVM
Número de características: 8
Características: ['lawobey', 'trstplt', 'trstprt', 'cntrytype', 'cluster', 'trst_happy', 'trstplt_mmx', 'trstprt_mmx']
AUC: 0.5974328714458839
Accuracy: 0.575
Precision: 0.47572203196347024
Recall: 0.575
F1-score: 0.44089727645215715


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import os

df_ideo = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_mmxeu.csv')
del df_ideo['lrscale'], df_ideo['lrscale_mmx']

def bestclasschi2(df, target_column, feature_range):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""
    best_metrics = {}

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=chi2, k=k)
            # Chi2 requiere entradas no negativas
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)
            y_pred = classifier.predict(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted')
            recall = recall_score(y_test, y_pred, average='weighted')
            f1 = f1_score(y_test, y_pred, average='weighted')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name
                best_metrics = {
                    'auc': auc,
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1
                }

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestClasschi2.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_metrics['auc']}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")
        f.write(f"Accuracy: {best_metrics['accuracy']}\n")
        f.write(f"Precision: {best_metrics['precision']}\n")
        f.write(f"Recall: {best_metrics['recall']}\n")
        f.write(f"F1-score: {best_metrics['f1']}\n")

    return best_model, best_k, best_features, best_classifier_name, best_metrics


best_model, best_k, best_features, best_classifier_name, best_metrics = bestclasschi2(df_ideo, 'ideologia', (8, 12))

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"AUC: {best_metrics['auc']}")
print(f"Accuracy: {best_metrics['accuracy']}")
print(f"Precision: {best_metrics['precision']}")
print(f"Recall: {best_metrics['recall']}")
print(f"F1-score: {best_metrics['f1']}")

c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 68
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 8
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 79
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 9
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

...

c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000025 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 101
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 11
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\xgboost\training.py:183: UserWarning: [06:31:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000118 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 112
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 12
[LightGBM] [Info] Start training from score -1,517923
[LightGBM] [Info] Start training from score -0,536143
[LightGBM] [Info] Start training from score -1,630491
Mejor modelo: SVM
Número de características: 12
Características: ['ppltrst', 'lawobey', 'stfgov', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'happy', 'cntrytype', 'cluster', 'trstprt_mmx']
AUC: 0.611868601935991
Accuracy: 0.5816666666666667
Precision: 0.5001597366911255
Recall: 0.5816666666666667
F1-score: 0.45326491568698574


c:\Users\Josue\4GA.Datascience\.venv2\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
